# FYP: Deep Learning for Jet Classification — Google Colab runner

Train the BDT / CNN / ParticleNet / Particle Transformer top-taggers on a **free Colab T4 GPU** (~30–50× faster than a laptop CPU), then download the results back to your Mac.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.

This notebook stores everything (data + checkpoints + plots) on **Google Drive** so that if Colab disconnects, you can just re-run and it **resumes from the last completed epoch** — no work lost.

Edit `GITHUB_USER` / `REPO` in the first code cell, then run cells top to bottom.

In [20]:
#@title 0. Settings — edit these
GITHUB_USER = "ramc77"          # <-- your GitHub username
REPO        = "fyp-jet-tagging" # <-- the repo name you pushed to
BRANCH      = "main"
# Where on Drive to keep persistent outputs (survives disconnects):
DRIVE_DIR   = "/content/drive/MyDrive/fyp_jet_tagging"
print('repo :', f'https://github.com/{GITHUB_USER}/{REPO}')
print('drive:', DRIVE_DIR)

repo : https://github.com/ramc77/fyp-jet-tagging
drive: /content/drive/MyDrive/fyp_jet_tagging


In [21]:
#@title 1. Check the GPU
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

Thu May 28 07:28:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
#@title 2. Mount Google Drive (for persistence across disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'data'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'results'), exist_ok=True)
print('Drive ready at', DRIVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready at /content/drive/MyDrive/fyp_jet_tagging


In [10]:
!git clone -b main https://github.com/ramc77/fyp-jet-tagging.git /content/fyp
%cd /content/fyp
!ls

Cloning into '/content/fyp'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 44 (delta 3), reused 44 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 889.36 KiB | 2.63 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/fyp
COLAB.md	  notebooks		  results		tools
config.py	  README.md		  run_full_pipeline.py
docs		  requirements-colab.txt  run_study.py
download_data.py  requirements.txt	  src


In [11]:
#@title 4. Install the Colab-only extras (keeps Colab's GPU torch)
!pip install -q -r requirements-colab.txt
print('done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 111.8 MB/s eta 0:00:00
done


In [12]:
#@title 5. Point data/ and results/ at Drive (persistent + resumable)
# Symlink the repo's data/ and results/ folders onto Drive so that
# (a) the 1.6 GB dataset is downloaded only once, ever, and
# (b) checkpoints survive a Colab disconnect.
import os
!rm -rf /content/fyp/data /content/fyp/results
os.symlink(os.path.join(DRIVE_DIR, 'data'),    '/content/fyp/data')
os.symlink(os.path.join(DRIVE_DIR, 'results'), '/content/fyp/results')
!ls -la /content/fyp/data /content/fyp/results

lrwxrwxrwx 1 root root 43 May 28 06:20 /content/fyp/data -> /content/drive/MyDrive/fyp_jet_tagging/data
lrwxrwxrwx 1 root root 46 May 28 06:20 /content/fyp/results -> /content/drive/MyDrive/fyp_jet_tagging/results


In [13]:
#@title 6. Download the dataset to Drive (only runs if not already there)
# Uses wget -c (resumable) which is more robust than urllib on Colab.
%%bash
cd /content/fyp/data
for f in train val test; do
  if [ -s "${f}.h5" ]; then
    echo "[skip] ${f}.h5 already present ($(du -h ${f}.h5 | cut -f1))"
  else
    echo "[get ] ${f}.h5"
    wget -c -q --show-progress "https://zenodo.org/records/2603256/files/${f}.h5"
  fi
done
ls -lh

[get ] train.h5
[get ] val.h5
[get ] test.h5
total 1.7G
-rw------- 1 root root 332M May 28 06:29 test.h5
-rw------- 1 root root 991M May 28 06:21 train.h5
-rw------- 1 root root 332M May 28 06:29 val.h5



     0K .......... .......... .......... .......... ..........  0%  133K 2h7m
    50K .......... .......... .......... .......... ..........  0%  499K 80m42s
   100K .......... .......... .......... .......... ..........  0%  820K 60m40s
   150K .......... .......... .......... .......... ..........  0% 9.29M 45m57s
   200K .......... .......... .......... .......... ..........  0%  327K 47m6s
   250K .......... .......... .......... .......... ..........  0% 36.4M 39m20s
   300K .......... .......... .......... .......... ..........  0% 5.74M 34m7s
   350K .......... .......... .......... .......... ..........  0% 48.9M 29m54s
   400K .......... .......... .......... .......... ..........  0%  545K 30m1s
   450K .......... .......... .......... .......... ..........  0%  871K 28m57s
   500K .......... .......... .......... .......... ..........  0% 26.4M 26m23s
   550K .......... .......... .......... .......... ..........  0% 64.1M 24m12s
   600K .......... .......... .......... ...

In [14]:
#@title 7a. (optional) Use the FULL 1.2M-jet dataset on GPU
# On a T4 the full dataset is feasible. Comment this cell out to keep the
# default 50k subset from config.py.
import re, pathlib
cfg = pathlib.Path('/content/fyp/config.py')
s = cfg.read_text()
s = re.sub(r'USE_SUBSET = True', 'USE_SUBSET = False', s)
cfg.write_text(s)
print('USE_SUBSET set to False (full dataset).')

USE_SUBSET set to False (full dataset).


In [18]:
import re, pathlib
cfg = pathlib.Path('/content/fyp/config.py')
s = cfg.read_text()
s = re.sub(r'USE_SUBSET\s*=\s*\w+',        'USE_SUBSET = True',      s)
s = re.sub(r'SUBSET_TRAIN\s*=\s*[\d_]+',   'SUBSET_TRAIN = 100_000', s)
s = re.sub(r'SUBSET_VAL\s*=\s*[\d_]+',     'SUBSET_VAL   = 50_000',  s)
s = re.sub(r'SUBSET_TEST\s*=\s*[\d_]+',    'SUBSET_TEST  = 50_000',  s)
cfg.write_text(s)
print('subset mode: 100k train / 50k val / 50k test')

subset mode: 100k train / 50k val / 50k test


In [24]:
%cd /content/fyp
!python run_full_pipeline.py

/content/fyp
╔══════════════════════════════════════════════════════════════╗
║  FYP: Deep Learning for Jet Classification                   ║
║  Top Quark Tagging in 14 TeV pp Collisions                   ║
║  BNBWU-CERN Collaboration                                    ║
╚══════════════════════════════════════════════════════════════╝

Device: cuda
Subset mode: True (train=100000)
Max particles: 100

████████████████████████████████████████████████████████████
  STEP 1: DATA PREPARATION
████████████████████████████████████████████████████████████

Processing train split...
Loading train data from /content/fyp/data/train.h5...
  Loaded 100000 jets | Signal: 50040 (50.0%) | Constituents: 200
  Computing jet substructure features...
  Creating jet images...
  Memory for train: 2826 MB

Processing val split...
Loading val data from /content/fyp/data/val.h5...
  Loaded 50000 jets | Signal: 25340 (50.7%) | Constituents: 200
  Computing jet substructure features...
  Creating jet images...
 

In [23]:
import pathlib
p = pathlib.Path('/content/fyp/src/particle_transformer.py')
s = p.read_text()
s = s.replace('attn_mask = ~mask.unsqueeze(1).unsqueeze(2)  # True where padded',
              'attn_mask = (mask == 0).unsqueeze(1).unsqueeze(2)  # True where padded')
p.write_text(s)
print('patched:', '(mask == 0)' in s)

patched: True


In [ ]:
#@title 8. Run the data-efficiency + calibration study (research angle)
!python run_study.py
!python tools/build_calibration_table.py

Device: cuda
Sizes : [5000, 10000, 25000, 50000]
Models: ['bdt', 'cnn', 'particlenet', 'transformer']

Processing train split...
Loading train data from /content/fyp/data/train.h5...
  Loaded 100000 jets | Signal: 50040 (50.0%) | Constituents: 200
  Computing jet substructure features...
  Creating jet images...
  Memory for train: 2826 MB

Processing val split...
Loading val data from /content/fyp/data/val.h5...
  Loaded 50000 jets | Signal: 25340 (50.7%) | Constituents: 200
  Computing jet substructure features...
  Creating jet images...
  Memory for val: 1413 MB

Processing test split...
Loading test data from /content/fyp/data/test.h5...
  Loaded 50000 jets | Signal: 24780 (49.6%) | Constituents: 200
  Computing jet substructure features...
  Creating jet images...
  Memory for test: 1413 MB

Computing normalization statistics from training set...

Data preparation complete!
  Training jets:   100000
  Validation jets: 50000
  Test jets:       50000

##############################

In [ ]:
#@title 9. Quick peek at the results
import json, glob, os
p = '/content/fyp/results/model_comparison.json'
if os.path.exists(p):
    print(json.dumps(json.load(open(p)), indent=2))
print('\nPlots produced:')
for f in sorted(glob.glob('/content/fyp/results/plots/*.pdf')):
    print('  ', os.path.basename(f))

In [ ]:
#@title 10. Zip results and download to your Mac
# (Everything is also already on your Drive under DRIVE_DIR/results.)
import shutil, os
out = '/content/fyp_results.zip'
if os.path.exists(out):
    os.remove(out)
shutil.make_archive('/content/fyp_results', 'zip', '/content/fyp/results')
print('zip size:', round(os.path.getsize(out)/1e6, 1), 'MB')
from google.colab import files
files.download(out)

## Back on your Mac

1. Unzip `fyp_results.zip` into the project's `results/` folder:
   ```bash
   cd /Users/ramchand/Desktop/Climate/May2026/FYP_particle_physics
   unzip -o ~/Downloads/fyp_results.zip -d results/
   ```
2. Re-compile the documents so they pick up the real figures + tables:
   ```bash
   cd docs/article  && pdflatex article.tex  && pdflatex article.tex
   cd ../tutorial   && pdflatex tutorial.tex && pdflatex tutorial.tex
   cd ../thesis     && pdflatex thesis.tex   && pdflatex thesis.tex
   ```

That's it — GPU does the heavy lifting, your Mac just renders the PDFs.